In [ ]:
!pip install langchain==0.3.25 langchain-groq langchain-community langchain-text-splitters langchain-huggingface chromadb pypdf wandb weave -q
print(" All packages installed!")

In [ ]:
import os
import wandb
import weave
import time

In [ ]:
# API keys
os.environ['GROQ_API_KEY'] = "your-groq-api-key"

# wandb key
wandb.login(key = "your-wandb-key")

# weave initialize
weave.init("model_comparison_monitoring")

print(" All keys set!")
print(" Weave monitoring activated!")

In [ ]:
from langchain_groq import ChatGroq
from langchain.schema import HumanMessage, SystemMessage
import pandas as pd
import time

print(" Imports successful!")

In [ ]:
from groq import Groq

client = Groq()
models_list = client.models.list()

for model in models_list.data:
    print(model.id)

In [ ]:
# Define 3 different models - all currently active on Groq!
models = {
    "LLaMA-3.1-8b": ChatGroq(
        model="llama-3.1-8b-instant",  # small and fast
        temperature=0.3
    ),
    "LLaMA-3.3-70b": ChatGroq(
        model="llama-3.3-70b-versatile",  # large and powerful
        temperature=0.3
    ),
    "Qwen3-32b": ChatGroq(
        model="qwen/qwen3-32b",  # medium, different architecture
        temperature=0.3
    )
}

print(" 3 models loaded!")
for name in models:
    print(f"  → {name}")

In [ ]:
@weave.op()
def compare_models(question: str):
    results = {}

    for model_name, model in models.items():
        print(f"⏳ Querying {model_name}...")

        start_time = time.time()

        # Send question to model
        messages = [
            SystemMessage(content="You are a helpful AI assistant. Give clear and concise answers."),
            HumanMessage(content=question)
        ]
        response = model.invoke(messages)

        end_time = time.time()

        # Store results
        results[model_name] = {
            "answer": response.content,
            "response_time": round(end_time - start_time, 2),
            "answer_length": len(response.content),
            "words": len(response.content.split())
        }

        print(f"   {model_name} → {results[model_name]['response_time']}s")

    return results

print(" Comparison function ready!")

In [ ]:
# Test questions - covering different types
questions = [
    "What is machine learning?",
    "Explain the difference between AI and ML",
    "What is a neural network?",
    "How does RAG work?",
    "What is the future of AI?"
]

all_results = []

for question in questions:
    print(f"\n{'='*60}")
    print(f" Question: {question}")
    print('='*60)

    results = compare_models(question)

    # Display results side by side
    for model_name, data in results.items():
        print(f"\n {model_name}:")
        print(f"   Time: {data['response_time']}s")
        print(f"    Words: {data['words']}")
        print(f"    Answer: {data['answer'][:150]}...")

    # Store for summary
    row = {"question": question}
    for model_name, data in results.items():
        row[f"{model_name}_time"] = data["response_time"]
        row[f"{model_name}_words"] = data["words"]
    all_results.append(row)

print("\n All questions compared across all models!")

In [ ]:
# Create summary dataframe
df = pd.DataFrame(all_results)

print("\n RESPONSE TIME COMPARISON (seconds):")
print("-" * 60)
time_cols = ["question"] + [f"{m}_time" for m in models.keys()]
print(df[time_cols].to_string(index=False))

print("\n ANSWER LENGTH COMPARISON (words):")
print("-" * 60)
words_cols = ["question"] + [f"{m}_words" for m in models.keys()]
print(df[words_cols].to_string(index=False))

# Find fastest model
avg_times = {
    model: df[f"{model}_time"].mean()
    for model in models.keys()
}
fastest = min(avg_times, key=avg_times.get)
most_detailed = max(
    models.keys(),
    key=lambda x: df[f"{x}_words"].mean()
)

print(f"\n FASTEST MODEL: {fastest} ({avg_times[fastest]:.2f}s avg)")
print(f" MOST DETAILED: {most_detailed}")
print(f"\n  Average response times:")
for model, avg in avg_times.items():
    print(f"   {model}: {avg:.2f}s")
print("\n Check W&B Weave dashboard for full traces!")